In [1]:
import os
import shutil

def clean_submission_folder(base_folder):

    # Step 1 — rename folders
    for name in os.listdir(base_folder):

        old_path = os.path.join(base_folder, name)

        if not os.path.isdir(old_path):
            continue

        new_name = name.split("_")[-1]
        new_path = os.path.join(base_folder, new_name)

        # avoid overwrite collisions
        if new_path != old_path:
            if os.path.exists(new_path):
                i = 1
                while os.path.exists(new_path + "_" + str(i)):
                    i += 1
                new_path = new_path + "_" + str(i)

            os.rename(old_path, new_path)

    # Step 2 — collect java files and flatten folders
    for folder in os.listdir(base_folder):

        folder_path = os.path.join(base_folder, folder)

        if not os.path.isdir(folder_path):
            continue

        java_files = []

        # find all java files recursively
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                if file.endswith(".java"):
                    java_files.append(os.path.join(root, file))

        # copy java files to top level
        for file in java_files:
            dest = os.path.join(folder_path, os.path.basename(file))

            if os.path.abspath(file) != os.path.abspath(dest):
                shutil.copy(file, dest)

        # remove all subfolders
        for item in os.listdir(folder_path):
            item_path = os.path.join(folder_path, item)

            if os.path.isdir(item_path):
                shutil.rmtree(item_path)

    print("Processing complete.")

In [2]:
# clean_submission_folder("/mnt/Personal/Projects/Micro_Codes/Java/PLC/Grading/submission/PLC2026SubmitAssignment1")

In [3]:
import os
import shutil
import subprocess
import csv
from tqdm import tqdm

def grade_submissions(grader_path, submissions_dir, output_csv):

    results = {}
    grader_name = "AssignmentGrader.java"

    students = [s for s in os.listdir(submissions_dir)
                if os.path.isdir(os.path.join(submissions_dir, s))]

    for username in tqdm(students, desc="Grading submissions"):

        student_folder = os.path.join(submissions_dir, username)

        try:
            shutil.copy(grader_path, os.path.join(student_folder, grader_name))

            compile_cmd = f'cd "{student_folder}" && javac *.java'
            compile_proc = subprocess.run(
                compile_cmd,
                shell=True,
                capture_output=True,
                text=True
            )

            if compile_proc.returncode != 0:

                compile_error = (compile_proc.stderr + compile_proc.stdout).strip()
                results[username] = [compile_error, "0", "0"]

            else:

                run_cmd = f'cd "{student_folder}" && java AssignmentGrader'

                try:

                    run_proc = subprocess.run(
                        run_cmd,
                        shell=True,
                        capture_output=True,
                        text=True,
                        timeout=5
                    )

                    grade_file = os.path.join(student_folder, "grade.txt")

                    if os.path.exists(grade_file):

                        with open(grade_file) as f:
                            lines = [l.strip() for l in f if l.strip()]

                        if len(lines) >= 2:
                            score = lines[-2]
                            percentage = lines[-1]
                            failures = " | ".join(lines[:-2])
                        else:
                            failures = " | ".join(lines)
                            score = "0"
                            percentage = "0"

                        results[username] = [failures, score, percentage]

                    else:
                        output = (run_proc.stderr + run_proc.stdout).strip()
                        results[username] = [output if output else "grade.txt not generated", "0", "0"]

                except subprocess.TimeoutExpired as e:

                    partial_output = ""

                    if e.stdout:
                        partial_output += e.stdout
                    if e.stderr:
                        partial_output += e.stderr

                    msg = "Execution timed out after 5 seconds"

                    if partial_output.strip():
                        msg += " | " + partial_output.strip()

                    results[username] = [msg, "0", "0"]

        finally:

            for file in ["AssignmentGrader.java", "grade.txt"]:
                path = os.path.join(student_folder, file)
                if os.path.exists(path):
                    os.remove(path)

            for f in os.listdir(student_folder):
                if f.endswith(".class"):
                    os.remove(os.path.join(student_folder, f))
                    
    results['agastya.ug2024'][0]+="isEmpty marked as isempty"
    results['agastya.ug2024'][2] = str(float(results['agastya.ug2024'][2][0:-1])-5)+"%"

    results['chinmayee.mcs2025'][0]+="Stack doesn't have a default constructor"
    results['chinmayee.mcs2025'][2] = str(float(results['chinmayee.mcs2025'][2][0:-1])-5)+"%"

    results['davincp.ug2024'][0]+="NumExpr(Expr e1, Expr e2, int value) should have been just NumExpr(int value)"
    results['davincp.ug2024'][2] = str(float(results['davincp.ug2024'][2][0:-1])-5)+"%"

    results['dhruvmp.mcs2025'][0]+="for (Token t : toklist) doesn't work if not set to java.lang.Iterable. Also class Iter was set to private instead of public."
    results['dhruvmp.mcs2025'][2] = str(float(results['dhruvmp.mcs2025'][2][0:-1])-10)+"%"

    # print(results)

    with open(output_csv, "w", newline="") as f:

        writer = csv.writer(f)
        writer.writerow(["username", "message", "score", "percentage"])

        for user, data in results.items():
            writer.writerow([user, data[0], data[1], data[2]])

    return results

In [4]:
grader_file = "/mnt/Personal/Projects/Micro_Codes/Java/PLC/Grading/testing/AssignmentGrader.java"
submissions_folder = "/mnt/Personal/Projects/Micro_Codes/Java/PLC/Grading/submission/all"
output_csv = "/mnt/Personal/Projects/Micro_Codes/Java/PLC/Grading/testing/results.csv"

# grade_submissions(grader_file, submissions_folder, output_csv)

In [5]:
def merge_grades(worksheet_path, grades_path, output_path):

    # load grades
    grades = {}
    with open(grades_path, "r", encoding="utf-8") as f:
        next(f)  # skip header
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split(",", 3)
            username = parts[0]
            percentage = parts[3].replace("%", "").strip()
            grades[username] = percentage


    def split_csv_line(line):
        fields = []
        field = ""
        in_quotes = False

        for c in line:
            if c == '"':
                in_quotes = not in_quotes
                field += c
            elif c == ',' and not in_quotes:
                fields.append(field)
                field = ""
            else:
                field += c

        fields.append(field)
        return fields


    with open(worksheet_path, "r", encoding="utf-8") as ws, \
         open(output_path, "w", encoding="utf-8") as out:

        header = ws.readline()
        out.write(header)

        for line in ws:
            original_line = line.rstrip("\n")

            fields = split_csv_line(original_line)

            email = fields[2]

            for username, perc in grades.items():
                if username in email:
                    fields[4] = perc
                    break

            new_line = ",".join(fields)
            out.write(new_line + "\n")

In [6]:
merge_grades(
    "/mnt/Personal/Projects/Micro_Codes/Java/PLC/Grading/Grades-PLC2026-Submit Assignment 1, 15 Feb 2026, due 25 Feb 2026-20605.csv",
    "/mnt/Personal/Projects/Micro_Codes/Java/PLC/Grading/testing/results.csv",
    "/mnt/Personal/Projects/Micro_Codes/Java/PLC/Grading/final_grade.csv"
)